# Previsao de Mortalidade Evitavel no SUS

Classificacao de obitos do SIM (DATASUS) como evitaveis ou nao-evitaveis.
Documentacao completa no README do repositorio: https://github.com/Weversson/Projeto_IML

---

## 1. Setup

Instale as dependencias e baixe os modelos treinados diretamente do GitHub Releases.

In [ ]:
!pip install -q pandas scikit-learn numpy joblib matplotlib seaborn

In [ ]:
# Opcional: clonar o repositorio e baixar os modelos
# Descomente as linhas abaixo se quiser usar os modelos ja treinados

# !git clone https://github.com/Weversson/Projeto_IML.git
# import os
# os.chdir('Projeto_IML')

# Baixar modelo treinado do GitHub Releases
# !mkdir -p models
# !curl -sL https://github.com/Weversson/Projeto_IML/releases/download/v1.0/evitavel_rf_2024_2025.pkl -o models/evitavel_rf_2024_2025.pkl
# !curl -sL https://github.com/Weversson/Projeto_IML/releases/download/v1.0/clusterizacao_kmeans.pkl -o models/clusterizacao_kmeans.pkl
# !curl -sL https://github.com/Weversson/Projeto_IML/releases/download/v1.0/scaler.pkl -o models/scaler.pkl

---

## 2. Download dos dados

Dados do SIM (DATASUS), formato JSON, direto do S3 oficial do Governo Federal.

In [ ]:
import os
import urllib.request

BASE_URL = "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SIM/json"
OUTPUT_DIR = "data/datasus"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for year in range(2025, 1995, -1):
    filename = f"Mortalidade_Geral_{year}.zip"
    filepath = os.path.join(OUTPUT_DIR, filename)
    url = f"{BASE_URL}/Mortalidade_Geral_{year}_json.zip"
    if not os.path.exists(filepath):
        print(f"Baixando {year}...")
        urllib.request.urlretrieve(url, filepath)
    else:
        print(f"{year} ja existe.")

print("Download concluido.")

---

## 3. Extracao e concatenacao

Arquivos de 1996-2023 vieram particionados. Sao concatenados em um JSON por ano.

In [ ]:
import glob

for year in range(2023, 1995, -1):
    patterns = [
        f"DO{str(year)[2:]}OPEN_*.json",
        f"Mortalidade_Geral_{year}_*.json"
    ]
    files = []
    for p in patterns:
        files.extend(sorted(glob.glob(os.path.join(OUTPUT_DIR, p))))

    if not files:
        continue

    out_file = os.path.join(OUTPUT_DIR, f"Mortalidade_Geral_{year}.json")
    with open(out_file, 'w') as outf:
        outf.write('[')
        for i, f in enumerate(files):
            with open(f, 'r') as inf:
                content = inf.read().strip()
            if content.startswith('['):
                content = content[1:]
            if content.endswith(']'):
                content = content[:-1]
            if i > 0:
                outf.write(',')
            outf.write(content)
            os.remove(f)
        outf.write(']')

    size_mb = os.path.getsize(out_file) / (1024 * 1024)
    print(f"{year}: {len(files)} arquivos -> {size_mb:.0f} MB")

print("Concatenacao concluida.")

---

## 4. Preprocessamento e target

Features demograficas + classificacao evitavel baseada em criterios OMS/MS.

In [ ]:
import pandas as pd
import numpy as np
import json

print("Carregando dados de 2024...")
with open("data/datasus/Mortalidade_Geral_2024.json") as f:
    data = json.load(f)

df = pd.DataFrame(data)
cols = ['IDADE', 'SEXO', 'RACACOR', 'ESTCIV', 'ESC2010', 'LOCOCOR', 'CAUSABAS', 'CODMUNRES', 'DTOBITO', 'HORAOBITO']
df = df[cols].copy()

# Filtragem
df = df[df['IDADE'].str.isdigit()]
df = df[df['SEXO'].isin(['1', '2'])]
df = df[df['RACACOR'].isin(['1', '2', '3', '4', '5'])]
df = df[df['CAUSABAS'].str.len() >= 3]

# Tipos
for c in ['IDADE', 'SEXO', 'RACACOR', 'LOCOCOR']:
    df[c] = df[c].astype(int)
df['ESC2010'] = pd.to_numeric(df['ESC2010'], errors='coerce').fillna(0).astype(int)
df['ESTCIV'] = pd.to_numeric(df['ESTCIV'], errors='coerce').fillna(0).astype(int)
df['UF'] = df['CODMUNRES'].str[:2].astype(int)
df['MES_OBITO'] = df['DTOBITO'].str[2:4].apply(
    lambda x: int(x) if x.isdigit() and 1 <= int(x) <= 12 else 0
)
df['HORA'] = df['HORAOBITO'].str[:2].apply(
    lambda x: int(x) if x.isdigit() and 0 <= int(x) <= 23 else -1
)
def faixa_etaria(i):
    if i <= 5: return 0
    elif i <= 15: return 1
    elif i <= 30: return 2
    elif i <= 45: return 3
    elif i <= 60: return 4
    elif i <= 75: return 5
    elif i <= 100: return 6
    else: return 7
df['FAIXA_ETARIA'] = df['IDADE'].apply(faixa_etaria)

# Target: evitavel
ci = {'I10','I11','I12','I13','I15','I20','I21','I22','I23','I24','I25','I60','I61','I62','I63','I64','I65','I66','I67','I69'}
cj = {'J00','J01','J02','J03','J04','J05','J06','J09','J10','J11','J12','J13','J14','J15','J16','J17','J18','J20','J21','J22','J40','J41','J42','J43','J44','J45','J46','J47'}
ce = {'E10','E11','E12','E13','E14'}
cn = {'P00','P01','P02','P03','P04','P05','P06','P07','P08','P09','P20','P21','P22','P23','P24','P25','P26','P27','P28','P29','P36','P37','P38','P39'}
def eh_ev(c):
    p, l = c[:3], c[0]
    return p in ci or p in cj or p in ce or l in 'VWXYZ' or l in 'AB' or p in cn

df['EVITAVEL'] = df['CAUSABAS'].apply(eh_ev).astype(int)

print(f"Registros: {len(df):,}")
print(f"Evitavel: {df['EVITAVEL'].mean()*100:.1f}%")
print(f"Nao-evitavel: {(1-df['EVITAVEL']).mean()*100:.1f}%")

---

## 5. Treinamento: Random Forest

Features: IDADE, SEXO, RACACOR, ESTCIV, ESC2010, LOCOCOR, UF, MES_OBITO, HORA, FAIXA_ETARIA.
CAUSABAS nao e utilizada como feature (decision intencional).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import joblib

fc = ['IDADE', 'SEXO', 'RACACOR', 'ESTCIV', 'ESC2010', 'LOCOCOR', 'UF', 'MES_OBITO', 'HORA', 'FAIXA_ETARIA']
X = df[fc].fillna(0)
y = df['EVITAVEL']

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Treino: {len(Xtr):,} | Teste: {len(Xte):,}")

rf = RandomForestClassifier(
    n_estimators=200, max_depth=15, min_samples_leaf=50,
    random_state=42, n_jobs=-1, class_weight='balanced'
)
rf.fit(Xtr, ytr)
yp = rf.predict(Xte)
yp2 = rf.predict_proba(Xte)[:, 1]

print(classification_report(yte, yp, target_names=['Nao-evitavel', 'Evitavel']))
print(f"AUC-ROC: {roc_auc_score(yte, yp2):.4f}")

In [ ]:
# Importancia das features
imp = pd.Series(rf.feature_importances_, index=fc).sort_values(ascending=False)
for f, i in imp.items():
    print(f"  {f:14s} {i:.4f} {'█' * int(i * 100)}")

---

## 6. Clustering (KMeans)

Identificacao de perfis naturais de mortalidade sem uso de rótulos.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

cc = ['IDADE', 'SEXO', 'RACACOR', 'ESTCIV', 'ESC2010', 'LOCOCOR', 'FAIXA_ETARIA']
sc = StandardScaler()
Xsc = sc.fit_transform(df[cc].fillna(0))

idx = np.random.choice(len(Xsc), min(100000, len(Xsc)), replace=False)
km = KMeans(n_clusters=6, random_state=42, n_init=10)
cl = km.fit_predict(Xsc[idx])

dc = df.iloc[idx].copy()
dc['CLUSTER'] = cl

for c in range(6):
    s = dc[dc['CLUSTER'] == c]
    t = s['CAUSABAS'].mode().iloc[0]
    print(f"Cluster {c} ({len(s):>6,}) | Idade: {s['IDADE'].mean():.0f} | M: {(s['SEXO']==1).mean()*100:.0f}% | Evit: {s['EVITAVEL'].mean()*100:.0f}% | Top: {t}")

---

## 7. Uso do modelo

Exemplo de classificacao de um novo obito.

In [ ]:
novo = pd.DataFrame({
    'IDADE': [65], 'SEXO': [1], 'RACACOR': [4], 'ESTCIV': [3],
    'ESC2010': [3], 'LOCOCOR': [1], 'UF': [35], 'MES_OBITO': [6],
    'HORA': [14], 'FAIXA_ETARIA': [5]
})

pred = rf.predict(novo)[0]
prob = rf.predict_proba(novo)[0]

print(f"Predicao: {'Evitavel' if pred == 1 else 'Nao-evitavel'}")
print(f"Prob. nao-evitavel: {prob[0]:.2%}")
print(f"Prob. evitavel:     {prob[1]:.2%}")

---

## 8. Salvar modelos

Os arquivos .pkl sao grandes (170 MB). Para compartilhar, faca upload como GitHub Release.

In [ ]:
import os

os.makedirs('models', exist_ok=True)
joblib.dump(rf, 'models/evitavel_rf_2024_2025.pkl')
joblib.dump(km, 'models/clusterizacao_kmeans.pkl')
joblib.dump(sc, 'models/scaler.pkl')

for f in os.listdir('models'):
    size = os.path.getsize(os.path.join('models', f))
    print(f"  {f}: {size / (1024*1024):.1f} MB")